# 01 — Project and Data Preparation

This notebook prepares and validates the Google Drive layout for the current text-assisted temporal action segmentation project.

Current main scope:

```text
Main dataset now: Breakfast
Next dataset later: Assembly101
```

This notebook is intentionally conservative:

- it creates the project/data/repository folder structure;
- it validates Breakfast in MS-TCN format;
- it checks 50Salads only if it is already present;

## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Imports and central paths


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import time
import numpy as np
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive')

PROJECT_ROOT = DRIVE_ROOT / 'mmf_tas_lab_project'
DATA_ROOT = DRIVE_ROOT / 'mmf_tas_lab_data'
EXTERNAL_ROOT = DRIVE_ROOT / 'mmf_tas_lab_external'

NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
SCRIPTS_DIR = PROJECT_ROOT / 'scripts'
CONFIGS_DIR = PROJECT_ROOT / 'configs'
DOCS_DIR = PROJECT_ROOT / 'docs'
RESULTS_DIR = PROJECT_ROOT / 'results'
ARCHIVE_DIR = PROJECT_ROOT / 'archive'

ZENODO_ROOT = DATA_ROOT / 'zenodo_ms_tcn_data'
BREAKFAST_ROOT = ZENODO_ROOT / 'breakfast'
FIFTY_SALADS_ROOT = ZENODO_ROOT / '50salads'

TEXT_ASSISTED_ROOT = DATA_ROOT / 'text_assisted_tas'
ASSEMBLY101_ROOT = DATA_ROOT / 'assembly101'
RAW_VIDEOS_ROOT = DATA_ROOT / 'raw_videos'

MSTCN_REPO = EXTERNAL_ROOT / 'ms-tcn'
LTCONTEXT_REPO = EXTERNAL_ROOT / 'ltcontext'
PROCEDUREVRL_REPO = EXTERNAL_ROOT / 'procedurevrl'
PGNET_REPO = EXTERNAL_ROOT / 'pgnet_legacy_optional'

LOCAL_RESULTS_DIR = Path('/content/mmf_tas_lab_project_results')
LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for path in [
    PROJECT_ROOT, NOTEBOOKS_DIR, SCRIPTS_DIR, CONFIGS_DIR, DOCS_DIR, RESULTS_DIR, ARCHIVE_DIR,
    DATA_ROOT, ZENODO_ROOT, TEXT_ASSISTED_ROOT, ASSEMBLY101_ROOT, RAW_VIDEOS_ROOT,
    EXTERNAL_ROOT, MSTCN_REPO.parent,
]:
    path.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('EXTERNAL_ROOT:', EXTERNAL_ROOT)
print('BREAKFAST_ROOT:', BREAKFAST_ROOT)


PROJECT_ROOT: /content/drive/MyDrive/mmf_tas_lab_project
DATA_ROOT: /content/drive/MyDrive/mmf_tas_lab_data
EXTERNAL_ROOT: /content/drive/MyDrive/mmf_tas_lab_external
BREAKFAST_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast


## 3. Configuration


In [ ]:
# Keep downloads disabled unless the data are missing
DOWNLOAD_ZENODO_MSTCN_DATA = False
ZENODO_MSTCN_DATA_URL = 'https://zenodo.org/records/3625992/files/data.zip?download=1'

# Current project target
STRICT_BREAKFAST_VALIDATION = True

# Legacy checks are optional. They will not stop the current Breakfast pipeline
CHECK_OPTIONAL_50SALADS_IF_PRESENT = True

print('DOWNLOAD_ZENODO_MSTCN_DATA:', DOWNLOAD_ZENODO_MSTCN_DATA)
print('STRICT_BREAKFAST_VALIDATION:', STRICT_BREAKFAST_VALIDATION)
print('CHECK_OPTIONAL_50SALADS_IF_PRESENT:', CHECK_OPTIONAL_50SALADS_IF_PRESENT)


DOWNLOAD_ZENODO_MSTCN_DATA: False
STRICT_BREAKFAST_VALIDATION: True
CHECK_OPTIONAL_50SALADS_IF_PRESENT: True


## 4. Helper functions


In [ ]:
def read_lines(path: Path):
    return path.read_text().splitlines()


def count_files(path: Path, pattern: str):
    return len(list(path.glob(pattern))) if path.exists() else 0


def require_exists(path: Path, description: str):
    if not path.exists():
        raise FileNotFoundError(f'Missing {description}: {path}')
    print(f'OK: {description}: {path}')
    return path


def non_backup_bundle_files(split_dir: Path):
    if not split_dir.exists():
        return []
    return sorted([p for p in split_dir.glob('*.bundle') if not p.name.endswith('~')])


def load_split(split_path: Path):
    ids = []
    for line in read_lines(split_path):
        line = line.strip()
        if line:
            ids.append(Path(line).stem)
    return ids


def load_mapping(mapping_path: Path):
    rows = []
    for line in read_lines(mapping_path):
        line = line.strip()
        if not line:
            continue
        idx, label = line.split(maxsplit=1)
        rows.append({'class_id': int(idx), 'label': label})
    return pd.DataFrame(rows).sort_values('class_id').reset_index(drop=True)


def inspect_feature(path: Path):
    x = np.load(path)
    if x.ndim != 2:
        raise ValueError(f'Expected 2D feature array [D, T], got shape {x.shape} for {path}')
    return {'shape': tuple(x.shape), 'dtype': str(x.dtype), 'feature_dim': int(x.shape[0]), 'timesteps': int(x.shape[1])}


def validate_mstcn_dataset(dataset_name: str, dataset_root: Path, expected_features=None, expected_gt=None, strict=False):
    report = {
        'dataset': dataset_name,
        'root': str(dataset_root),
        'exists': dataset_root.exists(),
        'features_dir': (dataset_root / 'features').exists(),
        'groundTruth_dir': (dataset_root / 'groundTruth').exists(),
        'splits_dir': (dataset_root / 'splits').exists(),
        'mapping_file': (dataset_root / 'mapping.txt').exists(),
        'num_features': count_files(dataset_root / 'features', '*.npy'),
        'num_groundTruth': count_files(dataset_root / 'groundTruth', '*.txt'),
        'num_split_files': len(non_backup_bundle_files(dataset_root / 'splits')),
        'num_classes': None,
        'missing_features_in_splits': None,
        'missing_labels_in_splits': None,
        'sample_feature_shape': None,
        'status': 'unchecked',
    }

    print('\n' + '=' * 70)
    print(f'Validating {dataset_name}')
    print('=' * 70)

    required = [
        dataset_root,
        dataset_root / 'features',
        dataset_root / 'groundTruth',
        dataset_root / 'splits',
        dataset_root / 'mapping.txt',
    ]

    if not all(p.exists() for p in required):
        report['status'] = 'missing_or_incomplete'
        print('Dataset is missing or incomplete. Skipping strict validation.')
        for key, value in report.items():
            print(f'{key}: {value}')
        if strict:
            missing = [str(p) for p in required if not p.exists()]
            raise FileNotFoundError(f'{dataset_name} is required but missing: {missing}')
        return report

    df_mapping = load_mapping(dataset_root / 'mapping.txt')
    report['num_classes'] = len(df_mapping)

    split_files = non_backup_bundle_files(dataset_root / 'splits')
    missing_features = []
    missing_labels = []

    for split_file in split_files:
        for video_id in load_split(split_file):
            if not (dataset_root / 'features' / f'{video_id}.npy').exists():
                missing_features.append(video_id)
            if not (dataset_root / 'groundTruth' / f'{video_id}.txt').exists():
                missing_labels.append(video_id)

    report['missing_features_in_splits'] = len(missing_features)
    report['missing_labels_in_splits'] = len(missing_labels)

    feature_files = sorted((dataset_root / 'features').glob('*.npy'))
    if feature_files:
        info = inspect_feature(feature_files[0])
        report['sample_feature_shape'] = str(info['shape'])

    problems = []
    if expected_features is not None and report['num_features'] != expected_features:
        problems.append(f'expected {expected_features} feature files, found {report["num_features"]}')
    if expected_gt is not None and report['num_groundTruth'] != expected_gt:
        problems.append(f'expected {expected_gt} groundTruth files, found {report["num_groundTruth"]}')
    if missing_features:
        problems.append(f'{len(missing_features)} split entries missing features')
    if missing_labels:
        problems.append(f'{len(missing_labels)} split entries missing labels')

    report['status'] = 'ready' if not problems else 'has_problems'

    for key, value in report.items():
        print(f'{key}: {value}')
    if problems:
        print('Problems:')
        for p in problems:
            print(' -', p)
        if strict:
            raise RuntimeError(f'{dataset_name} validation failed: {problems}')

    return report


def save_csv_local_then_drive(df: pd.DataFrame, local_path: Path, drive_path: Path):
    local_path.parent.mkdir(parents=True, exist_ok=True)
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(local_path, index=False)
    print('Saved local:', local_path)
    try:
        shutil.copy2(local_path, drive_path)
        print('Copied to Drive:', drive_path)
    except Exception as e:
        print('WARNING: Drive copy failed. Local file is safe:', local_path)
        print('Error:', repr(e))


## 5. Optional Zenodo download


In [ ]:
if DOWNLOAD_ZENODO_MSTCN_DATA:
    tmp_dir = Path('/content/tmp_zenodo_mstcn')
    zip_path = tmp_dir / 'data.zip'
    extract_dir = tmp_dir / 'extracted'
    tmp_dir.mkdir(parents=True, exist_ok=True)

    print('Downloading Zenodo MS-TCN data package...')
    subprocess.run(['wget', '-c', '-O', str(zip_path), ZENODO_MSTCN_DATA_URL], check=True)

    print('Extracting...')
    extract_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', str(zip_path), '-d', str(extract_dir)], check=True)

    source_data = extract_dir / 'data'
    require_exists(source_data, 'extracted Zenodo data directory')

    print('Copying to Google Drive...')
    subprocess.run(['cp', '-r', str(source_data) + '/.', str(ZENODO_ROOT)], check=True)
    print('Done:', ZENODO_ROOT)
else:
    print('Skipping download. Using existing data in Google Drive.')


Skipping download. Using existing data in Google Drive.


## 6. Validate Breakfast dataset


In [ ]:
breakfast_report = validate_mstcn_dataset(
    dataset_name='Breakfast',
    dataset_root=BREAKFAST_ROOT,
    expected_features=1712,
    expected_gt=1712,
    strict=STRICT_BREAKFAST_VALIDATION,
)

# Show mapping head and a few split files.
df_breakfast_mapping = load_mapping(BREAKFAST_ROOT / 'mapping.txt')
print('\nBreakfast classes:', len(df_breakfast_mapping))
display(df_breakfast_mapping.head(10))

print('\nBreakfast split files:')
for p in non_backup_bundle_files(BREAKFAST_ROOT / 'splits'):
    print(' ', p.name, 'videos:', len(load_split(p)))



Validating Breakfast
dataset: Breakfast
root: /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast
exists: True
features_dir: True
groundTruth_dir: True
splits_dir: True
mapping_file: True
num_features: 1712
num_groundTruth: 1712
num_split_files: 8
num_classes: 48
missing_features_in_splits: 0
missing_labels_in_splits: 0
sample_feature_shape: (2048, 832)
status: ready

Breakfast classes: 48


,class_id,label
0,0,SIL
1,1,pour_cereals
2,2,pour_milk
3,3,stir_cereals
4,4,take_bowl
5,5,pour_coffee
6,6,take_cup
7,7,spoon_sugar
8,8,stir_coffee
9,9,pour_sugar



Breakfast split files:
  test.split1.bundle videos: 252
  test.split2.bundle videos: 451
  test.split3.bundle videos: 433
  test.split4.bundle videos: 576
  train.split1.bundle videos: 1460
  train.split2.bundle videos: 1261
  train.split3.bundle videos: 1279
  train.split4.bundle videos: 1136


## 7. Optional 50Salads check


In [ ]:
if CHECK_OPTIONAL_50SALADS_IF_PRESENT and FIFTY_SALADS_ROOT.exists():
    fifty_salads_report = validate_mstcn_dataset(
        dataset_name='50Salads legacy optional',
        dataset_root=FIFTY_SALADS_ROOT,
        expected_features=50,
        expected_gt=50,
        strict=False,
    )
else:
    fifty_salads_report = {
        'dataset': '50Salads legacy optional',
        'root': str(FIFTY_SALADS_ROOT),
        'exists': FIFTY_SALADS_ROOT.exists(),
        'status': 'skipped_not_needed_for_current_scope',
    }
    print('Skipping 50Salads. It is optional for the current Breakfast proof-of-concept.')



Validating 50Salads legacy optional
dataset: 50Salads legacy optional
root: /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/50salads
exists: True
features_dir: True
groundTruth_dir: True
splits_dir: True
mapping_file: True
num_features: 50
num_groundTruth: 50
num_split_files: 10
num_classes: 19
missing_features_in_splits: 0
missing_labels_in_splits: 0
sample_feature_shape: (2048, 11679)
status: ready


## 8. Save preparation summary


In [ ]:
summary_rows = [breakfast_report, fifty_salads_report]
df_data_prep_summary = pd.DataFrame(summary_rows)
display(df_data_prep_summary)

local_summary_path = LOCAL_RESULTS_DIR / '01_data_preparation_summary.csv'
drive_summary_path = RESULTS_DIR / '01_data_preparation_summary.csv'
save_csv_local_then_drive(df_data_prep_summary, local_summary_path, drive_summary_path)

project_status = {
    'current_main_scope': ['Breakfast + MS-TCN-style teacher-student proof of concept'],
    'next_scope': ['full Breakfast split 1', 'Breakfast splits 1-4', 'Assembly101', 'LTContext'],
    'legacy_optional': ['50Salads', 'IKEA ASM', 'PGNet'],
    'breakfast_status': breakfast_report['status'],
    'drive_project_root': str(PROJECT_ROOT),
    'drive_data_root': str(DATA_ROOT),
}

local_status_path = LOCAL_RESULTS_DIR / '01_project_status.json'
drive_status_path = RESULTS_DIR / '01_project_status.json'
with local_status_path.open('w') as f:
    json.dump(project_status, f, indent=2)
try:
    shutil.copy2(local_status_path, drive_status_path)
except Exception as e:
    print('WARNING: could not copy project status to Drive:', repr(e))

print(json.dumps(project_status, indent=2))
print('\n01_data_preparation completed.')


,dataset,root,exists,features_dir,groundTruth_dir,splits_dir,mapping_file,num_features,num_groundTruth,num_split_files,num_classes,missing_features_in_splits,missing_labels_in_splits,sample_feature_shape,status
0,Breakfast,/content/drive/MyDrive/mmf_tas_lab_data/zenodo...,True,True,True,True,True,1712,1712,8,48,0,0,"(2048, 832)",ready
1,50Salads legacy optional,/content/drive/MyDrive/mmf_tas_lab_data/zenodo...,True,True,True,True,True,50,50,10,19,0,0,"(2048, 11679)",ready


Saved local: /content/mmf_tas_lab_project_results/01_data_preparation_summary.csv
Copied to Drive: /content/drive/MyDrive/mmf_tas_lab_project/results/01_data_preparation_summary.csv
{
  "current_main_scope": [
    "Breakfast + MS-TCN-style teacher-student proof of concept"
  ],
  "next_scope": [
    "full Breakfast split 1",
    "Breakfast splits 1-4",
    "Assembly101",
    "LTContext"
  ],
  "legacy_optional": [
    "50Salads",
    "IKEA ASM",
    "PGNet"
  ],
  "breakfast_status": "ready",
  "drive_project_root": "/content/drive/MyDrive/mmf_tas_lab_project",
  "drive_data_root": "/content/drive/MyDrive/mmf_tas_lab_data"
}

01_data_preparation completed.
